## DTB — Divisão Territorial Brasileira (IBGE) — Bronze

**Fonte:** [IBGE — Divisão Territorial Brasileira](https://www.ibge.gov.br/geociencias/organizacao-do-territorio/estrutura-territorial/23701-divisao-territorial-brasileira.html)

- **Licença:** dados públicos IBGE — uso livre com citação da fonte.
- **Atualização:** anual (edição 2025 inclui o município Boa Esperança do Norte/MT).
- **Coleta:** download de `DTB_2025.zip` no FTP do IBGE (`geoftp.ibge.gov.br/organizacao_do_territorio/divisao_territorial/2025/`), extração do arquivo `RELATORIO_DTB_BRASIL_2025_MUNICIPIOS.ods` e upload manual para o Volume `/Volumes/workspace/raw/IBGE/`.
- **Linhagem:** `DTB_2025.zip` → `/Volumes/workspace/raw/IBGE/RELATORIO_DTB_BRASIL_2025_MUNICIPIOS.ods` → `workspace.bronze.dtb`.
- **Grão bronze:** 1 linha por município — cópia fiel do relatório, apenas com nomes de colunas normalizados por `normalizar_colunas`.
- **Cabeçalho:** na linha 7 do ODS (`HEADER_ROW = 6`, índice 0-based). Colunas originais: UF | Nome_UF | Região Geográfica Intermediária | Nome Região Geográfica Intermediária | Região Geográfica Imediata | Nome Região Geográfica Imediata | Município | Código Município Completo | Nome_Município.
- **Atenção:** a coluna `UF` contém o CÓDIGO da UF (11–53), não a sigla; códigos são lidos como string limpa pelo helper `read_ods`.

In [0]:
%run ../shared/_setup

In [0]:
from data_pipeline import (
    read_ods,
    normalizar_colunas,
    save_table,
    add_column_comments
)
from catalogo.divisao_territorial_brasileira import DTB_COMMENTS

In [0]:
FILE_PATH = "/Volumes/workspace/raw/IBGE/RELATORIO_DTB_BRASIL_2025_MUNICIPIOS.ods"
TABLE_NAME = "workspace.bronze.dtb"
SHEET_NAME = 0  # primeira aba do ODS
HEADER_ROW = 6  # cabeçalho na linha 7 do relatório (índice 0-based)

In [0]:
df_raw = read_ods(spark, FILE_PATH, sheet_name=SHEET_NAME, header=HEADER_ROW)
print(f"Colunas originais ({len(df_raw.columns)}): {df_raw.columns}")
display(df_raw.limit(5))

In [0]:
df = normalizar_colunas(df_raw)
print(f"Colunas normalizadas: {df.columns}")
df.printSchema()
display(df.limit(5))

In [0]:
save_table(df, TABLE_NAME)
print(f"Tabela {TABLE_NAME} persistida: {spark.table(TABLE_NAME).count():,} linhas")

In [0]:
add_column_comments(
    spark,
    TABLE_NAME,
    DTB_COMMENTS
)

In [0]:
total_rows = df.count()
distinct_rows = df.distinct().count()
print(f"Total de registros: {total_rows:,}")
print(f"Registros únicos: {distinct_rows:,}")
print(f"Duplicatas: {total_rows - distinct_rows:,}")
display(spark.sql(f"DESCRIBE TABLE {TABLE_NAME}"))